In [1]:

from pprint import pprint

from ollama import chat
from tools import Top_Type, code_extraction, equi_check, linter, netlist
# from typing import TypedDict
# from langgraph.graph import START,StateGraph,END


In [2]:
Ollama_MODEL="qwen2.5-coder:7b"

# Checking for the presence of model and loading  the model 


In [3]:
stream=chat(model=Ollama_MODEL,
            messages=[{"role":"user","content":"What is your knowledge cut off and how  good are you with verilog and symbolic model verification coding ?"}],
            stream=True)
print("Printing the answer\n")
for chunk in stream:
    print(chunk['message']['content'],end='',flush=True)
print()

Printing the answer

My knowledge cut-off date is October 2023, and my capabilities are based on the data available up to that point. I can provide information and assistance on a wide range of topics, including programming languages like Verilog.

Verilog is a hardware description language (HDL) used for designing and verifying digital circuits. If you have specific questions or need help with Verilog code, feel free to ask! For symbolic model verification, which involves using mathematical models to prove properties about the design without simulating every possible execution, I can provide general guidance on tools and techniques commonly used in this area.

If you have a particular piece of Verilog code or a specific verification challenge, please share it, and I'll do my best to assist you!


In [4]:
_VERILOG_SYSTEM = """
ROLE:
Senior RTL designer; Verilog-2001 only.

INSTRUCTIONS:
Implement the user's specification exactly.
Generate clean, synthesizable RTL.
Use explicit signal widths and correct signedness.
Use always @(*) with '=' for combinational logic.
Use always @(posedge clk) with '<=' for sequential logic.
Assign all combinational outputs on every path.
Every case requires default.
Use $signed()/ $unsigned().

STEPS:
Analyze → generate → internally check syntax, widths,
signedness, latches, and synthesizability.

END:
Return ONLY complete Verilog-2001 code.
No markdown or explanation.

NEGATIVE:
No SystemVerilog, VHDL, logic, always_comb, always_ff,
typedef, enum, struct, interface, class, package,
assertions, dynamic arrays, declarations inside always blocks,
or expression/function-call slicing.
"""

In [5]:
_VERILOG_SYSTEM =""" 
ROLE
You are a senior RTL design engineer specializing in synthesizable Verilog-2001.

INSTRUCTIONS
Implement the user's specification exactly. Generate clean, synthesizable RTL.
Use Verilog-2001 only and return only the complete Verilog code.

STEPS

1. Identify all signals, their widths, signedness, clocks, resets, and required behavior.
2. Generate the RTL using correct sequential and combinational coding styles.
3. Before returning, check syntax, widths, constants, signedness, bit selections, and synthesis safety.

END GOAL
Produce Verilator/Icarus-compatible Verilog-2001 RTL with no width or syntax errors.

NARROW CONSTRAINTS

* Use explicit signal widths.
* Every sized numeric literal must have enough bits for all specified digits.
  Example: 3'b111, NOT 2'b111.
* Match constant widths to the destination/comparison signal where practical.
  Example: reg [2:0] count → 3'b111 or 3'd7.
* Never select bits outside a signal's declared range.
* Use '=' in always @(*) and '<=' in clocked always blocks.
* Give combinational outputs a value on every path.
* Include default in case statements.
* Use $signed/$unsigned when signedness matters.
* No SystemVerilog: logic, always_comb, always_ff, typedef, enum, struct, interface, class, etc.
* Do not invent requirements or modify the user's specification.
* Return ONLY Verilog-2001 code; no Markdown or explanation.

"""

In [ ]:
def code_gen(prompt:str)->str:
    """ 
    INPUT:
    prompt: excepts user promtp a string providing the detials about the verilog code to be written by the llm.

    OUTPTS:
        gives the code as output 
    """
    stream=chat(model=Ollama_MODEL,messages=[
                                                {
                                                    "role": "system",
                                                    "content": _VERILOG_SYSTEM
                                                },
                                                {
                                                    "role": "user",
                                                    "content": prompt
                                                }
                                            ],
                                            stream=True,
                                            options={
                                                    "temperature":0.5,
                                                     "seed":0
                                                     })
    resp=""
    for chunk in stream:
        content=chunk["message"]["content"]
        print(content,end="",flush=True)
        resp+=content
    return resp

In [7]:
User_inpt=input("Prompt the verilog module  requesting")
response=code_gen(User_inpt)

```verilog
module apb_completer (
    input wire clk,
    input wire rst_n,

    // APB Interface
    input wire [31:0] paddr,
    input wire [31:0] pwdata,
    input wire       penable,
    input wire       pwrite,
    output reg [31:0] prdata,
    output reg       pready,
    output reg       psel,

    // Internal Signals
    reg [31:0] data_reg,
    reg        write_en,
    reg        read_en
);

always @(posedge clk or negedge rst_n) begin
    if (!rst_n) begin
        prdata <= 32'b0;
        pready <= 1'b0;
        psel <= 1'b0;
        data_reg <= 32'b0;
        write_en <= 1'b0;
        read_en <= 1'b0;
    end else begin
        if (penable) begin
            psel <= 1'b1;
            pready <= 1'b1;

            if (pwrite) begin
                data_reg <= pwdata;
                write_en <= 1'b1;
                read_en <= 1'b0;
            end else begin
                prdata <= data_reg;
                write_en <= 1'b0;
                read_en <= 1'b1;
            end


In [8]:
clean_Vcode=code_extraction(response)
print(clean_Vcode)

module apb_completer (
    input wire clk,
    input wire rst_n,

    // APB Interface
    input wire [31:0] paddr,
    input wire [31:0] pwdata,
    input wire       penable,
    input wire       pwrite,
    output reg [31:0] prdata,
    output reg       pready,
    output reg       psel,

    // Internal Signals
    reg [31:0] data_reg,
    reg        write_en,
    reg        read_en
);

always @(posedge clk or negedge rst_n) begin
    if (!rst_n) begin
        prdata <= 32'b0;
        pready <= 1'b0;
        psel <= 1'b0;
        data_reg <= 32'b0;
        write_en <= 1'b0;
        read_en <= 1'b0;
    end else begin
        if (penable) begin
            psel <= 1'b1;
            pready <= 1'b1;

            if (pwrite) begin
                data_reg <= pwdata;
                write_en <= 1'b1;
                read_en <= 1'b0;
            end else begin
                prdata <= data_reg;
                write_en <= 1'b0;
                read_en <= 1'b1;
            end
        end

In [9]:
res=linter(clean_Vcode)
print(res)

{'return_code': 0, 'stdout': '', 'stderr': '%Warning-UNUSEDSIGNAL: /tmp/tmp0v8au6nc.v:6:23: Signal is not used: \'paddr\'\n                                              : ... note: In instance \'apb_completer\'\n    6 |     input wire [31:0] paddr,\n      |                       ^~~~~\n                       ... For warning description see https://verilator.org/warn/UNUSEDSIGNAL?v=5.020\n                       ... Use "/* verilator lint_off UNUSEDSIGNAL */" and lint_on around source to disable this message.\n'}


In [17]:
top=Top_Type(clean_Vcode)

In [18]:
pprint(top)

{'err': "/tmp/tmpz8beafe3.v:15: ERROR: Module port `\\data_reg' is neither "
        'input nor output.\n',
 'output': '\n'
           ' '
           '/----------------------------------------------------------------------------\\\n'
           ' |  yosys -- Yosys Open SYnthesis '
           'Suite                                       |\n'
           ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
           '<claire@yosyshq.com>         |\n'
           ' |  Distributed under an ISC-like license, type "license" to see '
           'terms        |\n'
           ' '
           '\\----------------------------------------------------------------------------/\n'
           ' Yosys 0.62+112 (git sha1 687a36af3, g++ 13.3.0-6ubuntu2~24.04.1 '
           '-fPIC -O3)\n'
           '\n'
           '-- Running command ` \n'
           '                             read_verilog /tmp/tmpz8beafe3.v;\n'
           '                             proc;\n'
           '                             sta

In [12]:
top_name=top['top_module']
print(top_name)

unknown_top


In [13]:
net_out=netlist(top=top_name,module_path=f"./src/{top_name}.v",lib_path="../Nangate45/Nangate45_typ.lib")

In [14]:
pprint(net_out)

{'netlist_returncode': 1,
 'netlist_stderr': "./src/unknown_top.v:15: ERROR: Module port `\\data_reg' is "
                   'neither input nor output.\n',
 'netlist_stdout': '\n'
                   ' '
                   '/----------------------------------------------------------------------------\\\n'
                   ' |  yosys -- Yosys Open SYnthesis '
                   'Suite                                       |\n'
                   ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
                   '<claire@yosyshq.com>         |\n'
                   ' |  Distributed under an ISC-like license, type "license" '
                   'to see terms        |\n'
                   ' '
                   '\\----------------------------------------------------------------------------/\n'
                   ' Yosys 0.62+112 (git sha1 687a36af3, g++ '
                   '13.3.0-6ubuntu2~24.04.1 -fPIC -O3)\n'
                   '\n'
                   '-- Running command `\n'
   

In [15]:
path_1=f"./src/{top_name}.v"
path_2=f"./src/{top_name}_netlist.v"
des_type=top["type"]
equi_out=equi_check(top=top_name,path_1=path_1,path_2=path_2,design_type=des_type)

In [16]:
pprint(equi_out)

{'pass': 'fail',
 'return_code': 1,
 'stderr': "./src/unknown_top.v:15: ERROR: Module port `\\data_reg' is neither "
           'input nor output.\n',
 'stdout': '\n'
           ' '
           '/----------------------------------------------------------------------------\\\n'
           ' |  yosys -- Yosys Open SYnthesis '
           'Suite                                       |\n'
           ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
           '<claire@yosyshq.com>         |\n'
           ' |  Distributed under an ISC-like license, type "license" to see '
           'terms        |\n'
           ' '
           '\\----------------------------------------------------------------------------/\n'
           ' Yosys 0.62+112 (git sha1 687a36af3, g++ 13.3.0-6ubuntu2~24.04.1 '
           '-fPIC -O3)\n'
           '\n'
           '-- Running command `\n'
           '            read_verilog ./src/unknown_top.v;\n'
           '            prep -top unknown_top;\n'
           '      